Hybrid Score = 0.3 x C-index + 0.7 x (1 - Weighted Brier Score)

## C-index (30%)
- Measures how well you rank fires by urgency
- Higher is better (0.5 to 1.0)
## Weighted Brier Score (70%)
- Measures calibration at 24h, 48h, 72h
- Uses censor-aware evaluation:
- Hits: 1 if hit by horizon H, else 0
- Censored after H: 0
- Censored before H: excluded
### Final weighted average: 0.3 x Brier@24h + 0.4 x Brier@48h + 0.3 x Brier@72h

In [ ]:
# using content/FE_train.csv to train random survival forest model and then generate predictions using content/FE_test.csv
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, brier_score # to evaluate based on Kaggle specifications

# load training data
train = pd.read_csv('content/FE_train.csv')
train = train.drop(columns=['event_id']) # we don't want to use event_id as a feature for prediction


In [ ]:
# split train data
train_df, val_df = train_test_split(
    train,
    test_size=0.2,
    stratify=train["event"], # want to keep event distribution balanced in both datasets
    random_state=42
)

# prepare data for random survival forest
X_train = train_df.drop(columns=["event", "time_to_hit_hours"])
y_train = Surv.from_dataframe("event", "time_to_hit_hours", train_df)

X_val = val_df.drop(columns=["event", "time_to_hit_hours"])
y_val = Surv.from_dataframe("event", "time_to_hit_hours", val_df)

In [ ]:
# calculating weighted average of concordance index and brier score for validation set

def compute_hybrid_score(rsf, X_train, y_train, X_val, y_val):
    rsf.fit(X_train, y_train) # fit model on training data

    risk_pred = rsf.predict(X_val) # get risk predictions for validation set
    c_index = concordance_index_censored(y_val["event"], y_val["time_to_hit_hours"], risk_pred)[0] # calculate concordance index

    # need full survival function predictions to calculate brier score
    surv_funcs = rsf.predict_survival_function(X_val)

    max_model_time = surv_funcs[0].domain[1]
    max_val_time = y_val["time_to_hit_hours"].max()
    max_model_time = min(max_model_time, max_val_time) # make sure we don't evaluate brier score at time horizons that are greater than the maximum time in the validation set or the maximum time that the model can predict for since that will lead to errors when calculating brier score
    # brier score requires a strictly less than upper bound and not less than or equal to upper bound so we need to make sure all of our horizons are less than max_model_time
    max_model_time = max_model_time - 1e-6 # subtract a small value from max_model_time to make sure all horizons are less than max_model_time
    
    horizons = np.array([24, 48, min(72, max_model_time)]) # time horizons to evaluate at

    # convert survival functions to matrix of survival predictions so we can use in brier_score
    surv_preds = np.asarray([
        [fn(t) for t in horizons]
        for fn in surv_funcs
    ])

    weights = np.array([0.3, 0.4, 0.3]) # weights for each time horizon
    _, brier_scores = brier_score(y_train, y_val, surv_preds, horizons) # calculate brier scores at each time horizon by unpacking the two output arrays (only need the one that gives us the scores)
    weighted_brier = np.sum(weights * brier_scores) # calculate weighted average brier score

    hybrid = 0.3 * c_index + 0.7 * (1 - weighted_brier) # calculate hybrid score

    # print(f"c-index = {c_index}")
    # print(f"weighted brier score = {weighted_brier}")
    # print(f"hybrid score = {hybrid}")
    return hybrid

In [4]:
# parameter fine tuning
param_grid = {
    "n_estimators": [200, 300, 400, 500, 600],
    "max_depth": [None, 10, 20, 30],
    "min_samples_leaf": [1, 3, 5, 10],
    "min_samples_split": [2, 5, 10, 20, 25],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7]
}

best_score = -np.inf
best_params = None
for n_estimators in param_grid["n_estimators"]:
    for max_depth in param_grid["max_depth"]:
        for min_samples_leaf in param_grid["min_samples_leaf"]:
            for min_samples_split in param_grid["min_samples_split"]:
                for max_features in param_grid["max_features"]:
                    rsf = RandomSurvivalForest(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        min_samples_split=min_samples_split,
                        max_features=max_features,
                        n_jobs=-1,
                        random_state=42
                    )
                    score = compute_hybrid_score(rsf, X_train, y_train, X_val, y_val)
                    print(f"n_estimators={n_estimators}, max_depth={max_depth}, min_samples_leaf={min_samples_leaf}, min_samples_split={min_samples_split}, max_features={max_features} → {score:.4f}")
                    if score > best_score:
                        best_score = score
                        best_params = {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "min_samples_leaf": min_samples_leaf,
                            "min_samples_split": min_samples_split,
                            "max_features": max_features
                        }
print("Best score:", best_score)
print("Best params:", best_params)

n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=2, max_features=sqrt → 0.9530
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=2, max_features=log2 → 0.9581
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=2, max_features=0.3 → 0.9660
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=2, max_features=0.5 → 0.9672
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=2, max_features=0.7 → 0.9650
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=5, max_features=sqrt → 0.9578
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=5, max_features=log2 → 0.9580
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=5, max_features=0.3 → 0.9651
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=5, max_features=0.5 → 0.9652
n_estimators=200, max_depth=None, min_samples_leaf=1, min_samples_split=5, max_features

In [5]:
# create RSF model
rsf = RandomSurvivalForest(
    n_estimators=300, 
    max_depth=None,
    min_samples_split=25, 
    min_samples_leaf=5, 
    max_features=0.7, 
    n_jobs=-1, 
    random_state=42
) # can play around with each of these values later to see what leads to the best performance

# fit model
rsf.fit(X_train, y_train)

,n_estimators,300
,max_depth,None
,min_samples_split,25
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,0.7
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-1
,random_state,42


In [6]:
# using content/FE_test.csv  
test = pd.read_csv('content/FE_test.csv')
X_test = test.drop(columns=['event_id']) # we don't want to use event_id as a feature for prediction
test_survival_predictions = rsf.predict_survival_function(X_test) # this will give us the predicted survival function for each observation in our training data
# aka gives us probability that fire i has NOT hit by time t
# so the probability that fire i HAS hit is 1 - survival_predictions[i](t) for each time t

In [7]:
times = [12, 24, 48, 72] # the hours we need to calculate the probability of wildfire hitting within
max_time = train["time_to_hit_hours"].max() # gives us 66.99447413277778

# this lets us know that we have no data on wildfires hitting after 66.99447413277778 hours, so we can only calculate probabilities for times up to that point
# meaning that we can't directly calculate probabilities for 72 hours, but we can still calculate for 12, 24, and 48 hours since those are all less than the max time in our training data
# will need to handle 72 hour time 
# for now, instead of calculating probability for 72 hours, calculate the probability for 66.99... hours and use that as our estimate instead

test_probabilities = []
for fn in test_survival_predictions:
    prob = []
    for t in times:
        t = min(t, max_time) # if t is greater than max_time, use max_time instead
        prob.append(1 - fn(t)) 
    test_probabilities.append(prob)
print(test_probabilities)

[[np.float64(0.003724121557454718), np.float64(0.0082908883674353), np.float64(0.0082908883674353), np.float64(0.013494950834460728)], [np.float64(0.5563664781117317), np.float64(0.8732352967170633), np.float64(0.9039139886896795), np.float64(0.9975)], [np.float64(0.002057454890788013), np.float64(0.0066242217007687065), np.float64(0.0066242217007687065), np.float64(0.011828284167794023)], [np.float64(0.5567505949208487), np.float64(0.8707265462083129), np.float64(0.9018062353319262), np.float64(0.996904761904762)], [np.float64(0.18989439105937278), np.float64(0.19060487545435068), np.float64(0.19060487545435068), np.float64(0.19060487545435068)], [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)], [np.float64(0.005390788224121312), np.float64(0.01120755503410198), np.float64(0.01120755503410198), np.float64(0.016411617501127407)], [np.float64(0.5515905344747587), np.float64(0.8615798305873325), np.float64(0.8972478202992464), np.float64(0.991111111111111)], [np.float

In [8]:
# doing checks on test_probabilities to make sure they look reasonable
print(len(test_probabilities) == len(test)) # just making sure we have a probability for each row in test data
# check that all probabilities are between 0 and 1
for prob in test_probabilities:
    for p in prob:
        if p < 0 or p > 1:
            print("Probability out of bounds:", p)
# making sure monotonicity enforced row-wise (prob_12h <= prob_24h <= prob_48h <= prob_72h)
for prob in test_probabilities:
    for i in range(1, len(prob)):
        if prob[i] < prob[i-1]:
            print("Monotonicity violated:", prob)

True


In [9]:
# now we want to take our probabilities and put it into CSV with one row per event_id and four probability columns: event_id, prob_12h, prob_24h, prob_48h, prob_72h
output_df = pd.DataFrame()
output_df['event_id'] = test['event_id']
output_df['prob_12h'] = [p[0] for p in test_probabilities]
output_df['prob_24h'] = [p[1] for p in test_probabilities]
output_df['prob_48h'] = [p[2] for p in test_probabilities]
output_df['prob_72h'] = [p[3] for p in test_probabilities]
output_df.to_csv('content/submission4.csv', index=False)